# Bài Thực Hành Chương 4 & 5: Tính Toán Số Học Nâng Cao với Thư Viện NumPy

### Mục tiêu học tập (CLO2, CLO4, CLO5):
- Khởi tạo mảng NumPy 1D, 2D, thuộc tính mảng (`shape`, `dtype`, `ndim`).
- Các phép toán số học vector hóa (Vectorized math & Broadcasting).
- Các hàm thống kê mô tả (`mean`, `std`, `min`, `max`, `percentile`).
- Đại số tuyến tính: Chuẩn vector (`np.linalg.norm`), Khoảng cách Euclid (Euclidean distance), Ma trận tương quan (`np.corrcoef`).
- Ứng dụng thực tế: Chuẩn hóa Z-Score và phân nhóm chỉ số RFM khách hàng.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'NumPy Version: {np.__version__}')


## 1. Khởi Tạo Mảng & Thao Tác Cơ Bản với NumPy


In [ ]:
# Khởi tạo mảng và kiểm tra thuộc tính
arr1d = np.array([10.5, 20.0, 35.2, 50.0, 100.0])
arr2d = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

print(f'arr1d - Shape: {arr1d.shape}, ndim: {arr1d.ndim}, dtype: {arr1d.dtype}')
print(f'arr2d - Shape: {arr2d.shape}, ndim: {arr2d.ndim}')
print('Đường chéo chính (Diagonal):', np.diag(arr2d))


## 2. Ứng Dụng Vector Hóa: Tính Toán RFM & Khoảng Cách Euclid (Customer Analytics)
Chúng ta sử dụng dữ liệu `clean_master_orders.csv` để tổng hợp và tính toán ma trận RFM bằng NumPy.


In [ ]:
orders_path = project_root / 'data' / 'processed' / 'clean_master_orders.csv'
df_orders = pd.read_csv(orders_path, parse_dates=['ordered_at'])

# Gom nhóm khách hàng
ref_date = df_orders['ordered_at'].max() + pd.Timedelta(days=1)
rfm_raw = df_orders.groupby('customer_id').agg(
    last_date=('ordered_at', 'max'),
    freq=('order_id', 'nunique'),
    monetary=('net_sales', 'sum')
).reset_index()

rfm_raw['recency'] = (ref_date - rfm_raw['last_date']).dt.days

# Chuyển đổi thành ma trận 2D NumPy (N x 3)
R = rfm_raw['recency'].to_numpy(dtype=np.float64)
F = rfm_raw['freq'].to_numpy(dtype=np.float64)
M = np.clip(rfm_raw['monetary'].to_numpy(dtype=np.float64), 0.01, None)

# Áp dụng Log transformation (NumPy ufunc)
log_R = np.log1p(R)
log_F = np.log1p(F)
log_M = np.log1p(M)

X = np.column_stack((log_R, log_F, log_M))
print(f'Kích thước ma trận đặc trưng X: {X.shape}')


## 3. Thống Kê Ma Trận & Chuẩn Hóa Z-Score qua Broadcasting (Chapter 5.5)


In [ ]:
# Tính Mean và Standard Deviation theo từng cột (axis=0)
means = np.mean(X, axis=0)
stds = np.std(X, axis=0)

print('Giá trị trung bình từng chiều (Log R, Log F, Log M):', np.round(means, 3))
print('Độ lệch chuẩn từng chiều:', np.round(stds, 3))

# Chuẩn hóa Z-Score: X_scaled = (X - Mean) / Std
X_scaled = (X - means) / stds

print('Kiểm tra Mean sau chuẩn hóa (gần bằng 0):', np.round(np.mean(X_scaled, axis=0), 5))
print('Kiểm tra Std sau chuẩn hóa (bằng 1):', np.round(np.std(X_scaled, axis=0), 5))


## 4. Đại Số Tuyến Tính & Khoảng Cách Euclid tới Nhóm Khách Hàng VIP (Chapter 5.6)


In [ ]:
# Điểm centroid của nhóm khách hàng lý tưởng (VIP): Recency cực thấp, Frequency & Monetary cực cao
vip_centroid = np.array([np.min(X_scaled[:, 0]), np.max(X_scaled[:, 1]), np.max(X_scaled[:, 2])])
print('Vector tọa độ VIP Centroid:', np.round(vip_centroid, 3))

# Tính khoảng cách Euclid giữa 19,868 khách hàng và VIP Centroid bằng np.linalg.norm
diff_matrix = X_scaled - vip_centroid
euclidean_distances = np.linalg.norm(diff_matrix, axis=1)

print(f'Khoảng cách ngắn nhất tới VIP: {np.min(euclidean_distances):.3f}')
print(f'Khoảng cách trung bình: {np.mean(euclidean_distances):.3f}')
print(f'Khoảng cách xa nhất: {np.max(euclidean_distances):.3f}')


## 5. Tính Ma Trận Tương Quan bằng `np.corrcoef`


In [ ]:
corr_matrix = np.corrcoef(X_scaled, rowvar=False)
print('Ma trận hệ số tương quan giữa [Log Recency, Log Frequency, Log Monetary]:')
print(np.round(corr_matrix, 3))
